# Deep Learning for Time Series

[Back to lesson](https://ml-viz-ruby.vercel.app/courses/time-series/03-deep-learning-for-time-series)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
np.random.seed(42)

## Sliding window: converting a series to supervised learning

In [ ]:
def sliding_windows(series, window, horizon=1):
    X, y = [], []
    for i in range(len(series) - window - horizon + 1):
        X.append(series[i:i+window])
        y.append(series[i+window:i+window+horizon])
    return np.array(X), np.array(y)

t = np.linspace(0, 8 * np.pi, 200)
series = np.sin(t) + 0.1 * np.random.randn(200)
X, y = sliding_windows(series, window=24, horizon=1)
print(f"Series length: {len(series)}")
print(f"Sliding windows — X shape: {X.shape}, y shape: {y.shape}")

## Walk-forward validation

In [ ]:
def walk_forward_mae(series, window=12, horizon=1, n_folds=5):
    n = len(series)
    fold_size = (n - window) // n_folds
    errors = []
    for i in range(n_folds):
        train_end = window + i * fold_size
        test_start = train_end
        test_end = min(test_start + fold_size, n)
        train = series[:train_end]
        test = series[test_start:test_end]
        pred = np.full(len(test), train[-1])
        errors.append(np.mean(np.abs(test - pred)))
    return np.array(errors)

t = np.linspace(0, 8 * np.pi, 200)
series = np.sin(t) + 0.1 * np.random.randn(200)
errors = walk_forward_mae(series)
print("Walk-forward MAE per fold:", errors.round(3))
print(f"Mean MAE: {errors.mean():.3f}")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, len(errors)+1), errors, color='#818cf8', alpha=0.8)
ax.set_xlabel('Fold', color='white'); ax.set_ylabel('MAE', color='white')
ax.set_title('Walk-Forward Validation MAE by Fold', color='white')
ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

## Simple manual RNN

In [ ]:
class SimpleRNN:
    def __init__(self, input_size, hidden_size, output_size, seed=42):
        rng = np.random.default_rng(seed)
        s = 0.1
        self.Wx = rng.normal(0, s, (hidden_size, input_size))
        self.Wh = rng.normal(0, s, (hidden_size, hidden_size))
        self.bh = np.zeros(hidden_size)
        self.Wy = rng.normal(0, s, (output_size, hidden_size))
        self.by = np.zeros(output_size)
    
    def forward(self, x_seq):
        h = np.zeros(self.Wh.shape[0])
        for x in x_seq:
            h = np.tanh(self.Wx @ x + self.Wh @ h + self.bh)
        return self.Wy @ h + self.by, h

t = np.linspace(0, 8 * np.pi, 200)
series = np.sin(t) + 0.1 * np.random.randn(200)
rnn = SimpleRNN(input_size=1, hidden_size=16, output_size=1)
window_sample = series[0:12].reshape(-1, 1)
pred, h_final = rnn.forward(window_sample)
print(f"Prediction: {pred[0]:.4f}, True: {series[12]:.4f}")
print(f"Hidden state norm: {np.linalg.norm(h_final):.4f}")

## Evaluation metrics

In [ ]:
t2 = np.linspace(0, 8 * np.pi, 200)
s2 = np.sin(t2) + 0.1 * np.random.randn(200)
true_vals = s2[150:]
naive_pred = np.full(len(true_vals), s2[149])

def evaluate(true, pred, name):
    mae = np.mean(np.abs(true - pred))
    rmse = np.sqrt(np.mean((true - pred)**2))
    mape = np.mean(np.abs((true - pred) / (np.abs(true) + 1e-8))) * 100
    print(f"{name:20s}  MAE={mae:.3f}  RMSE={rmse:.3f}  MAPE={mape:.1f}%")

evaluate(true_vals, naive_pred, "Naive (last value)")

## Your turn: Sliding window count

In [ ]:
# TODO(you): For a series of length 100, window=20, horizon=1,
# how many (X, y) samples does sliding_windows produce?
# Formula: len(series) - window - horizon + 1

n_samples = None  # replace with integer

assert n_samples is not None
assert n_samples == 80, f"Expected 80, got {n_samples}"
print(f"Correct: {n_samples} samples")

<details><summary>Solution</summary>

```python
n_samples = 100 - 20 - 1 + 1  # = 80
```
</details>